## Supply-chain — signing, SBOMs & the hardened service

The final layer isn't about the running container but about **trusting the image itself**: was it really built by your CI, and what's inside it?

**Image signing** answers "did this come from us?" **Cosign** (from the Sigstore project) is the modern standard — sign with a key or keyless GitHub OIDC, and the signature is stored in the registry against the image's digest:

```bash
cosign sign   --key cosign.key ghcr.io/myorg/api:1.2.3
cosign verify --key cosign.pub ghcr.io/myorg/api:1.2.3
```

Kubernetes admission controllers (Kyverno, Connaisseur) can then **refuse unsigned images**. (The older `DOCKER_CONTENT_TRUST=1` / Notary v1 path still exists but is largely superseded by Cosign.)

**SBOMs and provenance** answer "what's in it, and how was it built?" An **SBOM** (Software Bill of Materials — SPDX or CycloneDX) lists every package, version, and license; a **provenance attestation** records the Dockerfile, source commit, base image, and build args. BuildKit emits both:

```bash
docker buildx build --sbom=true --provenance=true -t myorg/api:1.2.3 --push .
```

Both attach as separate artifacts referenced from the manifest, so a consumer can scan or verify without running the image — and **SLSA** is the framework that grades how trustworthy that build chain is.

**Putting it all together — the hardened service**, every module-08 idea in one Compose block:

```yaml
services:
  api:
    image: ghcr.io/myorg/api@sha256:abc123...   # pinned by digest (module 07)
    user: "10001:10001"                          # non-root
    read_only: true
    tmpfs: [/tmp, /var/run]
    cap_drop: [ALL]
    cap_add: [NET_BIND_SERVICE]                  # least privilege
    security_opt: [no-new-privileges:true]       # no escalation
    secrets: [db_password]                       # file, not env
    restart: unless-stopped
```

Signed and SBOM'd upstream, pinned by digest, non-root, read-only, no capabilities, no escalation, secrets as files — a container you can defend in a review. That's the whole module in one manifest.